# **Linear Regression - Car Price Prediction**
نفس خطوات مشروع الـ Advertising، لكن على داتا سيارات (CAR_DETAILS_FROM_CAR_DEKHO.csv)
الهدف: نتنبأ بسعر بيع العربية (selling_price)

## 1) استيراد المكتبات

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2) تحميل البيانات

In [ ]:
data = pd.read_csv("/content/CAR_DETAILS_FROM_CAR_DEKHO.csv")


## 3) نظرة أولى على البيانات

In [ ]:
data.head()


In [ ]:
data.shape


In [ ]:
data.sample(5)


In [ ]:
data.tail()


## 4) هيكل البيانات وأنواعها

In [ ]:
data.info()


In [ ]:
data.dtypes


## 5) الملخص الإحصائي

In [ ]:
data.describe()


In [ ]:
data.describe(include='all')


## 6) القيم الفارغة (Missing Values)

In [ ]:
data.isnull().sum()


In [ ]:
data.isnull().sum() / len(data) * 100


## 7) التكرار (Duplicates)

In [ ]:
data.duplicated().sum()


In [ ]:
data[data.duplicated()]


In [ ]:
data.drop_duplicates(inplace=True)


## 8) القيم الفريدة (Unique Values / Cardinality)

In [ ]:
data.nunique()


In [ ]:
data['fuel'].value_counts()


In [ ]:
data['fuel'].unique()


In [ ]:
data['owner'].value_counts()


In [ ]:
data['seller_type'].value_counts()


In [ ]:
data['transmission'].value_counts()


## 9) فحص المتغير الهدف (selling_price)

In [ ]:
data['selling_price'].describe()


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(data['selling_price'], kde=True)
plt.title('Distribution of Selling Price')
plt.show()


## 10) الارتباط (Correlation)
هنعمل الارتباط على الأعمدة الرقمية بس (year, km_driven, selling_price)

In [ ]:
data[['year','km_driven','selling_price']].corr()


In [ ]:
data[['year','km_driven','selling_price']].corr()['selling_price'].sort_values(ascending=False)


## 11) فحص الـ Outliers

In [ ]:
data[['selling_price','km_driven']].boxplot()


In [ ]:
Q1 = data['selling_price'].quantile(0.25)
Q3 = data['selling_price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data['selling_price'] < lower_bound) | (data['selling_price'] > upper_bound)]
print(outliers)


In [ ]:
# حذف الـ outliers بتاعة selling_price
data = data[(data['selling_price'] >= lower_bound) & (data['selling_price'] <= upper_bound)]


In [ ]:
Q1 = data['km_driven'].quantile(0.25)
Q3 = data['km_driven'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data['km_driven'] < lower_bound) | (data['km_driven'] > upper_bound)]
print(outliers)


In [ ]:
# حذف الـ outliers بتاعة km_driven
data = data[(data['km_driven'] >= lower_bound) & (data['km_driven'] <= upper_bound)]


## 12) تجهيز الأعمدة النصية (Encoding)
الموديل بيفهم أرقام بس، فلازم نحول الأعمدة النصية (fuel, seller_type, transmission, owner) لأرقام باستخدام One-Hot Encoding (get_dummies)

In [ ]:
data = pd.get_dummies(data, columns=['fuel', 'seller_type', 'transmission', 'owner'], drop_first=True)
data.head()


## 13) تجهيز X و Y

In [ ]:
# بنشيل name (نص حر مالوش لازمة للموديل دلوقتي) وselling_price (الهدف) من X
X = data.drop(columns=['name', 'selling_price'])
X


In [ ]:
Y = data["selling_price"]
Y


## 14) تقسيم البيانات (Train / Test Split)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")


## 15) توحيد المقياس (Scaling)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 16) تدريب الموديل (Linear Regression)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)


## 17) تقييم الموديل بالـ MSE

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
